Setup

In [1]:
import os, shutil, subprocess, sys, torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | {props.total_memory / 1e9:.2f} GB")

for path in ["/kaggle/working", "/kaggle/tmp", "/tmp"]:
    if os.path.exists(path):
        total, used, free = shutil.disk_usage(path)
        print(f"Disk {path}: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

java_check = subprocess.run(["java", "-version"], capture_output=True, text=True)
print("Java status:", "OK" if java_check.returncode == 0 else "MISSING")
if java_check.stderr:
    print(" ", java_check.stderr.splitlines()[0])

Python: 3.13.16
PyTorch: 2.11.0+cu130 | CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.64 GB
  GPU 1: Tesla T4 | 15.64 GB
Disk /kaggle/working: 20.94 GB free / 20.96 GB total
Disk /tmp: 2700.00 GB free / 8758.39 GB total
Java status: OK
  openjdk version "21.0.12.1" 2026-08-18


In [2]:
import os

os.makedirs("/kaggle/tmp/hf_cache", exist_ok=True)
os.environ["HF_HOME"] = "/kaggle/tmp/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U peft accelerate huggingface_hub wandb pycocoevalcap easyocr rapidocr onnxruntime open-clip-torch

import transformers, peft, accelerate
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        val = secrets.get_secret(name)
        print(f"Secret {name}: FOUND (len={len(val)})")
    except Exception as e:
        print(f"Secret {name}: MISSING ({type(e).__name__})")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
transformers: 5.20.2
peft: 0.22.0
accelerate: 1.16.0
Secret HF_TOKEN: FOUND (len=37)
Secret WANDB_API_KEY: MISSING (BackendError)


BLIP

In [3]:
import time
import torch
from PIL import Image, ImageDraw, ImageFont
from transformers import Blip2Processor, Blip2ForConditionalGeneration

# Create a crisp synthetic test image with visible text (no network dependency)
img = Image.new("RGB", (512, 512), color=(240, 235, 225))
draw = ImageDraw.Draw(img)
draw.rectangle([120, 90, 392, 430], fill=(200, 45, 45), outline=(40, 40, 40), width=4)
draw.rectangle([145, 160, 367, 320], fill=(255, 255, 255))
font = ImageFont.load_default()
draw.text((165, 190), "ORGANIC\nTOMATO SOUP\nNET WT 14 OZ", fill=(20, 20, 20), font=font)

model_id = "Salesforce/blip2-opt-2.7b"
t0 = time.time()
processor = Blip2Processor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},
)
model.eval()
print(f"Model loaded in {time.time() - t0:.1f}s")
print(f"Allocated after load: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

prompt = "Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:"
inputs = processor(images=img, text=prompt, return_tensors="pt").to("cuda:0", torch.float16)

print("input_ids shape:", inputs["input_ids"].shape)
print("decoded input_ids:", repr(processor.tokenizer.decode(inputs["input_ids"][0])))

torch.cuda.reset_peak_memory_stats(0)
t1 = time.time()
with torch.no_grad():
    out = model.generate(
        **inputs,
        num_beams=5,
        max_new_tokens=60,
        no_repeat_ngram_size=3,
    )
elapsed = time.time() - t1

full_text = processor.batch_decode(out, skip_special_tokens=True)[0]
new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
print(f"Generated output shape: {out.shape} | Time: {elapsed:.2f}s | Peak VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")
print("Raw decoded output:", repr(full_text))

processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/882 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.56M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Model loaded in 96.6s
Allocated after load: 7.70 GB
input_ids shape: torch.Size([1, 55])
decoded input_ids: '<image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image><image></s>Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text:'
Generated output shape: torch.Size([1, 75]) | Time: 5.65s | Peak VRAM: 7.85 GB
Raw decoded output: 'Visible text: ORGANIC TOMATO SOUP NET WT 14 OZ. Alt-text: This soup is vegan, gluten-free, non-GMO, and made with organic ingredients\n'


loRA

In [4]:
import torch
from peft import LoraConfig, get_peft_model

# Freeze vision + Q-Former explicitly
for p in model.vision_model.parameters():
    p.requires_grad = False
for p in model.qformer.parameters():
    p.requires_grad = False
model.query_tokens.requires_grad = False
for p in model.language_projection.parameters():
    p.requires_grad = False

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model.language_model = get_peft_model(model.language_model, lora_cfg)
model.language_model.print_trainable_parameters()

# Verify LoRA weights are fp32 while base weights are fp16
lora_dtypes = {p.dtype for n, p in model.named_parameters() if p.requires_grad}
print("Trainable param dtypes:", lora_dtypes)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.train()

# Build 1 training example: prompt + target caption + EOS, prompt tokens masked to -100
prompt_text = "Visible text: ORGANIC TOMATO SOUP. Alt-text:"
target_text = " A red can of organic tomato soup with a white label."

prompt_enc = processor(images=img, text=prompt_text, return_tensors="pt")
full_enc = processor(
    images=img,
    text=prompt_text + target_text + processor.tokenizer.eos_token,
    return_tensors="pt",
)

input_ids = full_enc["input_ids"].to("cuda:0")
attention_mask = full_enc["attention_mask"].to("cuda:0")
pixel_values = full_enc["pixel_values"].to("cuda:0", torch.float16)

labels = input_ids.clone()
prompt_len = prompt_enc["input_ids"].shape[1]
labels[:, :prompt_len] = -100  # mask prompt (and any image placeholder tokens)

torch.cuda.reset_peak_memory_stats(0)
scaler = torch.amp.GradScaler("cuda")

with torch.amp.autocast("cuda", dtype=torch.float16):
    outputs = model(
        pixel_values=pixel_values,
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )
    loss = outputs.loss

scaler.scale(loss).backward()
print(f"Dry-run loss: {loss.item():.4f} (finite={torch.isfinite(loss).item()})")
print(f"Peak VRAM during 1-step train: {torch.cuda.max_memory_allocated(0)/1e9:.2f} GB")

model.zero_grad(set_to_none=True)

trainable params: 5,242,880 || all params: 2,656,921,600 || trainable%: 0.1973
Trainable param dtypes: {torch.float32}


/usr/local/lib/python3.13/dist-packages/peft/import_utils.py:195: UserWarning: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported. torchao support in PEFT is disabled; if you want to use a torchao-quantized model with PEFT, please upgrade torchao.
  warnings.warn(
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Dry-run loss: 2.4327 (finite=True)
Peak VRAM during 1-step train: 7.83 GB


OCR Benchmark

In [5]:
import time
import numpy as np
from PIL import Image, ImageDraw, ImageFilter

# Make a slightly blurred & tilted label image to test OCR realism
ocr_img = Image.new("RGB", (400, 260), (235, 230, 220))
d = ImageDraw.Draw(ocr_img)
d.text((40, 60), "TYLENOL Extra Strength\n500 mg Acetaminophen\n100 Caplets", fill=(15, 15, 15))
ocr_img = ocr_img.rotate(6, expand=True, fillcolor=(235, 230, 220)).filter(ImageFilter.GaussianBlur(0.6))
ocr_np = np.array(ocr_img)

# 1. EasyOCR (CPU + GPU)
import easyocr

t0 = time.time()
reader_cpu = easyocr.Reader(["en"], gpu=False, verbose=False)
init_easy_cpu = time.time() - t0
t1 = time.time()
res_easy_cpu = reader_cpu.readtext(ocr_np)
lat_easy_cpu = time.time() - t1

t0 = time.time()
reader_gpu = easyocr.Reader(["en"], gpu=True, verbose=False)
init_easy_gpu = time.time() - t0
t1 = time.time()
res_easy_gpu = reader_gpu.readtext(ocr_np)
lat_easy_gpu = time.time() - t1

print(f"[EasyOCR CPU] init={init_easy_cpu:.2f}s | infer={lat_easy_cpu:.3f}s")
print("  texts:", [(t, round(c, 2)) for _, t, c in res_easy_cpu])
print(f"[EasyOCR GPU] init={init_easy_gpu:.2f}s | infer={lat_easy_gpu:.3f}s")

# 2. RapidOCR (ONNXRuntime CPU - reflects free HF Space UX)
from rapidocr import RapidOCR

t0 = time.time()
rapid = RapidOCR()
init_rapid = time.time() - t0
t1 = time.time()
res_rapid = rapid(ocr_np)
lat_rapid = time.time() - t1

print(f"[RapidOCR CPU] init={init_rapid:.2f}s | infer={lat_rapid:.3f}s")
if res_rapid and getattr(res_rapid, "txts", None):
    print("  texts:", [(t, round(float(s), 2)) for t, s in zip(res_rapid.txts, res_rapid.scores)])
else:
    print("  raw:", res_rapid)

[EasyOCR CPU] init=2.65s | infer=1.160s
  texts: [('TYLENOL Extra Stretath', np.float64(0.35)), ('500 mot', np.float64(0.23)), ('100 Caplets', np.float64(0.98)), ('~oaminophan', np.float64(0.05))]
[EasyOCR GPU] init=2.37s | infer=1.137s


[INFO] 2026-11-20 19:17:07,777 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 19:17:08,046 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-20 19:17:08,047 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-11-20 19:17:09,159 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 19:17:09,166 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-20 19:17:09,167 [RapidOCR] main.py:63: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-11-20 19:17:09,209 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-11-20 19:17:09,287 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

[RapidOCR CPU] init=0.02s | infer=1.731s
  texts: [('TYLENOL Extra Strength', 1.0), ('500 mg Acetaminophen', 0.98), ('100 Caplets', 1.0)]


VizWiz captions

In [6]:
import subprocess, zipfile
from pathlib import Path

# PASTE YOUR CHALLENGE ANNOTATIONS GOOGLE DRIVE LINK HERE:
CHALLENGE_GDRIVE_URL = "https://drive.google.com/file/d/1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH/view?usp=sharing"

!pip install -q gdown

DATA_DIR = Path("/kaggle/tmp/vizwiz")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ANN_DIR = DATA_DIR / "annotations"
ANN_DIR.mkdir(parents=True, exist_ok=True)

# 1. Download Challenge Annotations from Google Drive (supports both file and folder URLs)
if "/folders/" in CHALLENGE_GDRIVE_URL:
    subprocess.run(
        ["gdown", "--folder", CHALLENGE_GDRIVE_URL, "-O", str(ANN_DIR)],
        check=True,
    )
else:
    ann_file = DATA_DIR / "annotations_dl"
    subprocess.run(
        ["gdown", "--fuzzy", CHALLENGE_GDRIVE_URL, "-O", str(ann_file)],
        check=True,
    )
    if zipfile.is_zipfile(ann_file):
        print("Extracting annotations zip ...")
        with zipfile.ZipFile(ann_file, "r") as zf:
            zf.extractall(ANN_DIR)
    else:
        # In case it downloaded a single JSON directly
        ann_file.rename(ANN_DIR / "downloaded_annotations.json")

# Also extract any nested zips inside ANN_DIR just in case
for zp in list(ANN_DIR.rglob("*.zip")):
    print("Extracting nested zip:", zp.name)
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(zp.parent)

# 2. Download & extract official train and val images into /kaggle/tmp/vizwiz
image_urls = {
    "train_images": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip",
    "val_images": "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip",
}

for name, url in image_urls.items():
    out_zip = DATA_DIR / f"{name}.zip"
    if not out_zip.exists():
        print(f"Downloading {name} (~3.5-10GB) ...")
        subprocess.run(["wget", "-q", "--show-progress", "-O", str(out_zip), url], check=True)
    extract_dir = DATA_DIR / name
    if not extract_dir.exists():
        print(f"Extracting {out_zip.name} ...")
        with zipfile.ZipFile(out_zip, "r") as zf:
            zf.extractall(extract_dir)

print("\nExtracted annotation files:")
for p in sorted(ANN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p} ({p.stat().st_size / 1e6:.2f} MB)")

Downloading...
From: https://drive.google.com/uc?id=1YODzc6H-OoKGyeA0ZrmXUbmU1jd64pTH
To: /kaggle/tmp/vizwiz/annotations_dl
100%|██████████| 5.59M/5.59M [00:00<00:00, 238MB/s]


Extracting annotations zip ...



     0K .......... .......... .......... .......... ..........  0% 2.02M 88m53s
    50K .......... .......... .......... .......... ..........  0% 4.03M 66m42s
   100K .......... .......... .......... .......... ..........  0%  140M 44m53s
   150K .......... .......... .......... .......... ..........  0% 4.03M 44m48s
   200K .......... .......... .......... .......... ..........  0%  164M 36m4s
   250K .......... .......... .......... .......... ..........  0%  286M 30m9s
   300K .......... .......... .......... .......... ..........  0% 4.12M 32m4s
   350K .......... .......... .......... .......... ..........  0%  152M 28m12s
   400K .......... .......... .......... .......... ..........  0%  163M 25m12s
   450K .......... .......... .......... .......... ..........  0% 4.27M 26m53s
   500K .......... .......... .......... .......... ..........  0%  171M 24m32s
   550K .......... .......... .......... .......... ..........  0% 77.2M 22m41s
   600K .......... .......... .......... .

Extracting train_images.zip ...



     0K .......... .......... .......... .......... ..........  0% 2.06M 26m54s
    50K .......... .......... .......... .......... ..........  0% 4.17M 20m6s
   100K .......... .......... .......... .......... ..........  0%  134M 13m32s
   150K .......... .......... .......... .......... ..........  0% 4.24M 13m26s
   200K .......... .......... .......... .......... ..........  0%  121M 10m50s
   250K .......... .......... .......... .......... ..........  0%  226M 9m4s
   300K .......... .......... .......... .......... ..........  0%  163M 7m49s
   350K .......... .......... .......... .......... ..........  0% 4.48M 8m23s
   400K .......... .......... .......... .......... ..........  0%  143M 7m30s
   450K .......... .......... .......... .......... ..........  0%  169M 6m47s
   500K .......... .......... .......... .......... ..........  0%  170M 6m12s
   550K .......... .......... .......... .......... ..........  0%  207M 5m42s
   600K .......... .......... .......... .......

Extracting val_images.zip ...

Extracted annotation files:
  /kaggle/tmp/vizwiz/annotations/__MACOSX/._annotations (0.00 MB)
  /kaggle/tmp/vizwiz/annotations/__MACOSX/annotations/._.DS_Store (0.00 MB)
  /kaggle/tmp/vizwiz/annotations/__MACOSX/annotations/._test.json (0.00 MB)
  /kaggle/tmp/vizwiz/annotations/__MACOSX/annotations/._train.json (0.00 MB)
  /kaggle/tmp/vizwiz/annotations/__MACOSX/annotations/._val.json (0.00 MB)
  /kaggle/tmp/vizwiz/annotations/annotations/.DS_Store (0.01 MB)
  /kaggle/tmp/vizwiz/annotations/annotations/test.json (8.63 MB)
  /kaggle/tmp/vizwiz/annotations/annotations/train.json (25.08 MB)
  /kaggle/tmp/vizwiz/annotations/annotations/val.json (8.34 MB)


Dataset audit

In [7]:
import json, statistics
from collections import Counter, defaultdict
from pathlib import Path

DATA_DIR = Path("/kaggle/tmp/vizwiz")

# Locate train.json and val.json inside extracted annotations
ann_files = {p.name: p for p in (DATA_DIR / "annotations").rglob("*.json")}
print("Annotation JSONs:", list(ann_files.keys()))

train_json_path = ann_files.get("train.json")
val_json_path = ann_files.get("val.json")
assert train_json_path and val_json_path, f"Expected train.json and val.json, got {ann_files}"

def find_image_dir(base: Path) -> Path:
    jpgs = list(base.rglob("*.jpg"))
    assert jpgs, f"No jpgs found under {base}"
    return jpgs[0].parent

train_img_dir = find_image_dir(DATA_DIR / "train_images")
val_img_dir = find_image_dir(DATA_DIR / "val_images")
print("Train img dir:", train_img_dir, "| count:", len(list(train_img_dir.glob("*.jpg"))))
print("Val img dir:  ", val_img_dir,   "| count:", len(list(val_img_dir.glob("*.jpg"))))

def audit_split(json_path: Path, img_dir: Path, split_name: str):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    images = data["images"]
    anns = data["annotations"]
    img_by_id = {im["id"]: im for im in images}

    anns_by_img = defaultdict(list)
    missing_files = 0
    for im in images:
        if not (img_dir / im["file_name"]).exists():
            missing_files += 1

    precanned_caps = 0
    rejected_caps = 0
    valid_word_lens = []
    cr_count = 0

    for a in anns:
        cap = a["caption"]
        if "\r" in cap:
            cr_count += 1
        anns_by_img[a["image_id"]].append(a)
        if a.get("is_precanned", False):
            precanned_caps += 1
        elif a.get("is_rejected", False):
            rejected_caps += 1
        else:
            words = cap.strip().split()
            if words:
                valid_word_lens.append(len(words))

    # Image-level unclear & text_detected breakdowns
    maj_precanned_imgs = 0
    all_precanned_or_rejected_imgs = 0
    text_det_imgs = sum(1 for im in images if im.get("text_detected", False))

    for im in images:
        im_anns = anns_by_img[im["id"]]
        n_pre = sum(1 for a in im_anns if a.get("is_precanned", False))
        n_valid = sum(1 for a in im_anns if not a.get("is_precanned", False) and not a.get("is_rejected", False) and a["caption"].strip())
        if n_pre >= 3:  # majority of 5 captions
            maj_precanned_imgs += 1
        if n_valid == 0:
            all_precanned_or_rejected_imgs += 1

    valid_word_lens.sort()
    n_w = len(valid_word_lens)
    summary = {
        "split": split_name,
        "num_images": len(images),
        "num_annotations": len(anns),
        "missing_image_files": missing_files,
        "captions_with_carriage_return": cr_count,
        "precanned_captions": precanned_caps,
        "precanned_caption_pct": round(100 * precanned_caps / len(anns), 2),
        "rejected_captions": rejected_caps,
        "rejected_caption_pct": round(100 * rejected_caps / len(anns), 2),
        "images_text_detected": text_det_imgs,
        "images_text_detected_pct": round(100 * text_det_imgs / len(images), 2),
        "images_majority_precanned_ge3": maj_precanned_imgs,
        "images_majority_precanned_pct": round(100 * maj_precanned_imgs / len(images), 2),
        "images_zero_valid_captions": all_precanned_or_rejected_imgs,
        "valid_caption_words_mean": round(statistics.mean(valid_word_lens), 2),
        "valid_caption_words_median": valid_word_lens[n_w // 2],
        "valid_caption_words_p10": valid_word_lens[int(0.10 * n_w)],
        "valid_caption_words_p90": valid_word_lens[int(0.90 * n_w)],
        "valid_caption_words_max": valid_word_lens[-1],
    }
    return summary, data, anns_by_img

train_summary, train_raw, train_anns_by_img = audit_split(train_json_path, train_img_dir, "train")
val_summary, val_raw, val_anns_by_img = audit_split(val_json_path, val_img_dir, "val")

print("\n=== TRAIN AUDIT ===")
for k, v in train_summary.items(): print(f"  {k}: {v}")
print("\n=== VAL AUDIT ===")
for k, v in val_summary.items(): print(f"  {k}: {v}")

Annotation JSONs: ['test.json', 'val.json', 'train.json', '._train.json', '._val.json', '._test.json']
Train img dir: /kaggle/tmp/vizwiz/train_images/train | count: 23954
Val img dir:   /kaggle/tmp/vizwiz/val_images/val | count: 7750

=== TRAIN AUDIT ===
  split: train
  num_images: 23431
  num_annotations: 117155
  missing_image_files: 0
  captions_with_carriage_return: 0
  precanned_captions: 15159
  precanned_caption_pct: 12.94
  rejected_captions: 1421
  rejected_caption_pct: 1.21
  images_text_detected: 14701
  images_text_detected_pct: 62.74
  images_majority_precanned_ge3: 2134
  images_majority_precanned_pct: 9.11
  images_zero_valid_captions: 565
  valid_caption_words_mean: 12.04
  valid_caption_words_median: 11
  valid_caption_words_p10: 8
  valid_caption_words_p90: 17
  valid_caption_words_max: 113

=== VAL AUDIT ===
  split: val
  num_images: 7750
  num_annotations: 38750
  missing_image_files: 0
  captions_with_carriage_return: 0
  precanned_captions: 5167
  precanned_capt

HF Save

In [8]:
import json, random
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import train_test_split
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

ART_DIR = Path("/kaggle/working/artifacts")
ART_DIR.mkdir(parents=True, exist_ok=True)

# Build stratification key per train image: (text_detected, majority_precanned)
img_ids = []
strata = []
for im in train_raw["images"]:
    im_anns = train_anns_by_img[im["id"]]
    is_unclear = sum(1 for a in im_anns if a.get("is_precanned", False)) >= 3
    has_text = bool(im.get("text_detected", False))
    img_ids.append(im["id"])
    strata.append(f"text={int(has_text)}_unclear={int(is_unclear)}")

train_ids, dev_ids = train_test_split(
    img_ids,
    test_size=1000,
    random_state=42,
    stratify=strata,
)

# Verify zero overlap and matching distributions
assert set(train_ids).isdisjoint(set(dev_ids))
val_ids = [im["id"] for im in val_raw["images"]]

def strata_counts(id_list):
    id_set = set(id_list)
    c = defaultdict(int)
    for iid, st in zip(img_ids, strata):
        if iid in id_set:
            c[st] += 1
    return dict(c)

splits_payload = {
    "seed": 42,
    "train_ids": sorted(train_ids),
    "dev_ids": sorted(dev_ids),
    "val_ids": sorted(val_ids),
    "counts": {
        "train": len(train_ids),
        "dev": len(dev_ids),
        "val": len(val_ids),
    },
    "strata_train": strata_counts(train_ids),
    "strata_dev": strata_counts(dev_ids),
}

splits_path = ART_DIR / "splits_v1.json"
report_path = ART_DIR / "data_report_v1.json"

with open(splits_path, "w") as f:
    json.dump(splits_payload, f, indent=2)
with open(report_path, "w") as f:
    json.dump({"train": train_summary, "val": val_summary, "splits": splits_payload["counts"]}, f, indent=2)

print("Split counts:", splits_payload["counts"])
print("Train strata:", splits_payload["strata_train"])
print("Dev strata:  ", splits_payload["strata_dev"])

# Push frozen split & report to a private/public HF Hub dataset repo for persistence
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
api = HfApi(token=hf_token)
user = api.whoami()["name"]
REPO_ID = f"{user}/vizwiz-alttext-artifacts"
api.create_repo(repo_id=REPO_ID, repo_type="dataset", exist_ok=True, private=False)

for local_p in [splits_path, report_path]:
    api.upload_file(
        path_or_fileobj=str(local_p),
        path_in_repo=f"phase1/{local_p.name}",
        repo_id=REPO_ID,
        repo_type="dataset",
        commit_message="Phase 1: frozen VizWiz splits and data audit report",
    )
print(f"Uploaded Phase 1 artifacts to https://huggingface.co/datasets/{REPO_ID}")

Split counts: {'train': 22431, 'dev': 1000, 'val': 7750}
Train strata: {'text=1_unclear=0': 13783, 'text=0_unclear=0': 6605, 'text=0_unclear=1': 1753, 'text=1_unclear=1': 290}
Dev strata:   {'text=1_unclear=0': 615, 'text=1_unclear=1': 13, 'text=0_unclear=0': 294, 'text=0_unclear=1': 78}


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded Phase 1 artifacts to https://huggingface.co/datasets/B4BUK4/vizwiz-alttext-artifacts
